# # 🚀 DQN Training — Eco-Scale RL Summative
# Train DQN agent with 10 hyperparameter experiments on a Kubernetes autoscaling environment.


## Install dependencies


In [ ]:
# !pip install -q gymnasium==0.29.1 stable-baselines3==2.3.2 numpy matplotlib pandas torch tensorboard


## Environment definition (embedded)


In [ ]:
import gymnasium as gym
import numpy as np
import os
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

class KubernetesEnv(gym.Env):
    """Custom Gymnasium environment simulating a Kubernetes cluster for pod autoscaling."""
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 4}

    def __init__(self, trace_type="cyclical", render_mode=None, max_steps=288):
        super().__init__()
        self.trace_type = trace_type
        self.render_mode = render_mode
        self.max_steps = max_steps

        self.observation_space = gym.spaces.Box(
            low=np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 1.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )
        self.action_space = gym.spaces.Discrete(3)

        self.alpha = 0.5   # latency weight
        self.beta = 0.3    # wasted pods weight
        self.gamma_r = 0.2 # scaling cost weight

        self.pod_count = 3
        self.current_step = 0
        self.time_of_day = 0
        self.cpu_util = 0.1
        self.request_queue = 50
        self.latency = 0.0
        self.breach_count = 0
        self.episode_reward = 0.0
        self.latency_history = []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.pod_count = 3
        self.current_step = 0
        self.time_of_day = 0
        self.cpu_util = 0.1
        self.request_queue = 50
        self.latency = 0.0
        self.breach_count = 0
        self.episode_reward = 0.0
        self.latency_history = []
        return self._get_obs(), {}

    def step(self, action):
        if action == 0:
            self.pod_count = max(1, self.pod_count - 1)
        elif action == 2:
            self.pod_count = min(20, self.pod_count + 1)

        self.current_step += 1
        self.time_of_day = (self.time_of_day + 1) % 24

        if self.trace_type == "burst":
            self.cpu_util = self._get_burst_traffic(self.current_step)
        else:
            self.cpu_util = self._get_traffic(self.current_step)

        self.request_queue = int(self.cpu_util * 500)
        reward = self._calculate_reward(action)
        self.episode_reward += reward

        terminated = self._check_termination()
        if terminated:
            reward -= 10.0
        truncated = self.current_step >= self.max_steps

        obs = self._get_obs()
        info = {
            "latency": self.latency, "pods": self.pod_count,
            "step": self.current_step, "cpu_util": self.cpu_util,
            "request_queue": self.request_queue, "time_of_day": self.time_of_day,
            "episode_reward": self.episode_reward,
            "wasted_pods": self._get_wasted_pods(),
        }
        return obs, reward, terminated, truncated, info

    def _get_obs(self):
        return np.array([
            float(self.cpu_util), float(self.pod_count) / 20.0,
            float(self.request_queue) / 1000.0, float(self.time_of_day) / 23.0,
        ], dtype=np.float32)

    def _calculate_reward(self, action):
        self.latency = min(self.request_queue / (self.pod_count * 50.0), 1.0)
        self.latency_history.append(self.latency)
        wasted_pods = self._get_wasted_pods()
        scaling_cost = 1.0 if action != 1 else 0.0
        return -(self.alpha * self.latency) - (self.beta * wasted_pods) - (self.gamma_r * scaling_cost)

    def _get_wasted_pods(self):
        required_pods = max(1, int(np.ceil(self.request_queue / 50.0)))
        return max(0, self.pod_count - required_pods) / 20.0

    def _check_termination(self):
        if self.latency >= 1.0:
            self.breach_count += 1
        else:
            self.breach_count = 0
        return self.breach_count >= 3

    def _get_traffic(self, step):
        hour = step % 24
        if 6 <= hour <= 22:
            base = 0.3 + 0.5 * np.sin((hour - 6) * np.pi / 12)
        else:
            base = 0.1
        noise = self.np_random.normal(0, 0.05)
        return float(np.clip(base + noise, 0.05, 1.0))

    def _get_burst_traffic(self, step):
        base = self._get_traffic(step)
        if self.np_random.random() < 0.1:
            spike = self.np_random.uniform(0.4, 0.7)
            return float(np.clip(base + spike, 0.0, 1.0))
        return base


# Sanity check
env = KubernetesEnv()
obs, _ = env.reset()
print(f"✅ Environment ready. Obs shape: {obs.shape}, Obs: {obs}")
obs, r, t, tr, info = env.step(1)
print(f"   Step OK — Reward: {r:.4f}, Latency: {info['latency']:.3f}, Pods: {info['pods']}")


## DQN hyperparameter configs


In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback

CONFIGS = [
    dict(learning_rate=1e-4, gamma=0.99, buffer_size=10000, batch_size=64,
         exploration_fraction=0.3, exploration_final_eps=0.05, target_update_interval=100,
         notes="Baseline"),
    dict(learning_rate=1e-3, gamma=0.99, buffer_size=10000, batch_size=64,
         exploration_fraction=0.3, exploration_final_eps=0.05, target_update_interval=100,
         notes="Higher LR"),
    dict(learning_rate=1e-4, gamma=0.95, buffer_size=10000, batch_size=64,
         exploration_fraction=0.3, exploration_final_eps=0.05, target_update_interval=100,
         notes="Lower gamma"),
    dict(learning_rate=1e-4, gamma=0.99, buffer_size=50000, batch_size=64,
         exploration_fraction=0.3, exploration_final_eps=0.05, target_update_interval=100,
         notes="Larger buffer"),
    dict(learning_rate=1e-4, gamma=0.99, buffer_size=10000, batch_size=128,
         exploration_fraction=0.3, exploration_final_eps=0.05, target_update_interval=100,
         notes="Larger batch"),
    dict(learning_rate=1e-4, gamma=0.99, buffer_size=10000, batch_size=64,
         exploration_fraction=0.5, exploration_final_eps=0.05, target_update_interval=100,
         notes="More exploration"),
    dict(learning_rate=1e-4, gamma=0.99, buffer_size=10000, batch_size=64,
         exploration_fraction=0.1, exploration_final_eps=0.05, target_update_interval=100,
         notes="Less exploration"),
    dict(learning_rate=1e-4, gamma=0.99, buffer_size=10000, batch_size=64,
         exploration_fraction=0.3, exploration_final_eps=0.01, target_update_interval=100,
         notes="Lower final eps"),
    dict(learning_rate=1e-4, gamma=0.99, buffer_size=10000, batch_size=64,
         exploration_fraction=0.3, exploration_final_eps=0.05, target_update_interval=500,
         notes="Slower target update"),
    dict(learning_rate=5e-5, gamma=0.999, buffer_size=20000, batch_size=32,
         exploration_fraction=0.4, exploration_final_eps=0.02, target_update_interval=200,
         notes="Combined changes"),
]

TOTAL_TIMESTEPS = 100_000
EVAL_EPISODES = 10

os.makedirs("models/dqn", exist_ok=True)
os.makedirs("logs/dqn", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print(f"📋 {len(CONFIGS)} DQN configurations ready to train")


## Train all 10 DQN runs


In [ ]:
def evaluate_model(model, env, n_episodes=10):
    rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        total = 0.0
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(int(action))
            total += reward
            done = terminated or truncated
        rewards.append(total)
    return float(np.mean(rewards)), float(np.std(rewards))


results = []

for i, cfg in enumerate(CONFIGS, start=1):
    notes = cfg.pop("notes")
    print(f"\n{'='*60}")
    print(f"  DQN Run {i}/10 — {notes}")
    print(f"{'='*60}")

    env = KubernetesEnv(trace_type="cyclical")
    eval_env = KubernetesEnv(trace_type="cyclical")

    run_log = f"logs/dqn/run_{i}"
    eval_cb = EvalCallback(
        eval_env, best_model_save_path=f"models/dqn/run_{i}_best",
        log_path=run_log, eval_freq=5000, n_eval_episodes=5,
        deterministic=True, verbose=0,
    )

    model = DQN(policy="MlpPolicy", env=env, verbose=1,
                tensorboard_log=run_log, **cfg)
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_cb)
    model.save(f"models/dqn/eco_scale_dqn_run_{i}")

    mean_r, std_r = evaluate_model(model, eval_env, EVAL_EPISODES)
    print(f"  → Mean Reward: {mean_r:.2f} ± {std_r:.2f}")

    results.append({
        "Run": i, "learning_rate": cfg["learning_rate"],
        "gamma": cfg["gamma"], "buffer_size": cfg["buffer_size"],
        "batch_size": cfg["batch_size"],
        "exploration_fraction": cfg["exploration_fraction"],
        "exploration_final_eps": cfg["exploration_final_eps"],
        "target_update_interval": cfg["target_update_interval"],
        "Mean Reward": round(mean_r, 2), "Std Reward": round(std_r, 2),
        "Notes": notes,
    })
    cfg["notes"] = notes

df = pd.DataFrame(results)
df.to_csv("outputs/dqn_results.csv", index=False)
print(f"\n✅ DQN training complete! Results:")
print(df.to_string(index=False))


## Plot DQN results


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("DQN Hyperparameter Tuning Results", fontsize=14, fontweight="bold")

# Bar chart
colors = ["#4CAF50" if r == df["Mean Reward"].max() else "#2196F3" for r in df["Mean Reward"]]
ax1.bar(df["Run"], df["Mean Reward"], yerr=df["Std Reward"], color=colors, alpha=0.8, edgecolor="white")
ax1.set_xlabel("Run")
ax1.set_ylabel("Mean Episode Reward")
ax1.set_title("Mean Reward by Run")
ax1.set_xticks(df["Run"])
ax1.grid(axis="y", alpha=0.3)

# Horizontal detail chart
ax2.barh([f"R{r} ({n})" for r, n in zip(df["Run"], df["Notes"])],
         df["Mean Reward"], color=colors, alpha=0.8, edgecolor="white")
ax2.set_xlabel("Mean Episode Reward")
ax2.set_title("Hyperparameter Effect")
ax2.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/dqn_hyperparameter_tuning.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved to outputs/dqn_hyperparameter_tuning.png")


## Plot training curves from eval logs


In [ ]:
import glob

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title("DQN Training Curves (All Runs)", fontweight="bold")

for i in range(1, 11):
    eval_files = glob.glob(f"logs/dqn/run_{i}/evaluations.npz")
    if eval_files:
        data = np.load(eval_files[0])
        timesteps = data["timesteps"]
        mean_rewards = data["results"].mean(axis=1)
        ax.plot(timesteps, mean_rewards, label=f"Run {i}", alpha=0.7)

ax.set_xlabel("Timesteps")
ax.set_ylabel("Mean Eval Reward")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/dqn_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Training curves saved")


## Download models and results


In [ ]:
# Works on BOTH Colab and Kaggle
!zip -r dqn_output.zip models/dqn/ outputs/dqn_results.csv outputs/dqn_*.png

try:
    from google.colab import files
    files.download('dqn_output.zip')
    print('✅ Download started (Colab)')
except ImportError:
    print('✅ dqn_output.zip is ready!')
    print('   Kaggle: Click Save Version → Output tab to download')
